In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("spark-batch")\
    .master("local[*]")\
    .getOrCreate()

spark
    

In [ ]:
df = spark.read.format('csv').option('header', 'True').option('inferSchema', 'True').load('C:\\Users\\maniy\\Downloads\\beginner-assessment-b\\Beginner-Assessment-B\\spark-batch\\data\\raw\\transactions.csv')

In [ ]:
df.show()

+--------------+-----------+----------+-------+----------------+---------+-------+
|transaction_id|customer_id|product_id| amount|transaction_date|   status|channel|
+--------------+-----------+----------+-------+----------------+---------+-------+
|             1|         40|       372|1958.68|      2025-02-22|   failed| online|
|             2|      38259|       261|1749.76|      2025-07-16|   failed| online|
|             3|      36343|      4266| 368.51|      2025-12-23|completed| online|
|             4|      17193|      3434| 461.68|      2025-09-24|   failed| online|
|             5|      15598|      1315|1714.99|      2025-01-15|   failed| online|
|             6|      30455|         1|1593.34|      2025-03-21|completed| online|
|             7|      28844|         1|1956.29|      2025-11-09|   failed| online|
|             8|      42092|         1| 308.89|      2025-06-10|   failed| online|
|             9|      22427|       714| 746.83|      2025-07-01|completed| online|
|   

# Remove failed transactions

In [ ]:
df = df.filter(df.status != "failed")

In [ ]:
df.show()

+--------------+-----------+----------+-------+----------------+---------+-------+
|transaction_id|customer_id|product_id| amount|transaction_date|   status|channel|
+--------------+-----------+----------+-------+----------------+---------+-------+
|             3|      36343|      4266| 368.51|      2025-12-23|completed| online|
|             6|      30455|         1|1593.34|      2025-03-21|completed| online|
|             9|      22427|       714| 746.83|      2025-07-01|completed| online|
|            10|       4800|      4479|1224.87|      2025-09-20|completed| online|
|            11|      36506|      2124| 117.75|      2025-05-03|completed| online|
|            12|      20079|         1|1169.23|      2025-10-01|completed| online|
|            14|      17746|         1| 698.93|      2025-04-23|completed| online|
|            15|       6689|      1252|1402.57|      2025-09-27|completed| online|
|            16|      21048|         1| 884.64|      2025-11-30|completed| online|
|   

# Deduplicate records

In [ ]:
type(df)

pyspark.sql.dataframe.DataFrame

In [ ]:
df = df.dropDuplicates(["transaction_id"])

In [ ]:
df.show()

+--------------+-----------+----------+-------+----------------+---------+-------+
|transaction_id|customer_id|product_id| amount|transaction_date|   status|channel|
+--------------+-----------+----------+-------+----------------+---------+-------+
|             3|      36343|      4266| 368.51|      2025-12-23|completed| online|
|             6|      30455|         1|1593.34|      2025-03-21|completed| online|
|             9|      22427|       714| 746.83|      2025-07-01|completed| online|
|            12|      20079|         1|1169.23|      2025-10-01|completed| online|
|            15|       6689|      1252|1402.57|      2025-09-27|completed| online|
|            16|      21048|         1| 884.64|      2025-11-30|completed| online|
|            19|      20425|      3540|1353.08|      2025-07-22|completed| online|
|            28|      12644|         1|  69.18|      2025-01-04|completed| online|
|            31|      33799|      2132| 903.75|      2025-10-31|completed| online|
|   

In [ ]:
df.write \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .parquet("data/processed/transactions_parquet/")


Py4JJavaError: An error occurred while calling o45.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://wiki.apache.org/hadoop/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:735)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:270)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:286)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:978)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:660)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:700)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:672)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:699)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.ja